In [2]:
import os
import numpy as np
import librosa
import cv2
from tqdm import tqdm

# --- CONFIGURATION ---
INPUT_DIR = '/kaggle/input/release-in-the-wild/release_in_the_wild'
OUTPUT_DIR = '/kaggle/working/spectrogram_dataset'
SAMPLE_RATE = 16000  # Standard for voice
DURATION = 4         # Duration in seconds to crop/pad
N_MELS = 128         # Height of the image

def create_spectrogram(audio_path, save_path):
    try:
        # 1. Load Audio (load only necessary duration to save RAM)
        y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, duration=DURATION)
        
        # 2. Pad audio if shorter than DURATION
        target_length = DURATION * SAMPLE_RATE
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)), mode='constant')
        elif len(y) > target_length:
            y = y[:target_length]

        # 3. Generate Mel Spectrogram
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
        
        # 4. Convert to Log-Scale (dB)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        # 5. Normalize to 0-255 (for image saving)
        # Min-Max scaling
        img = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min())
        img = (img * 255).astype(np.uint8)
        
        # 6. Save using OpenCV
        # Flip vertically because librosa returns low freq at index 0 (top), 
        # but images usually have low freq at bottom. 
        img = np.flip(img, axis=0) 
        cv2.imwrite(save_path, img)
        
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")

# --- MAIN LOOP ---
# Walk through train, val, test folders
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.endswith(('.wav', '.mp3', '.flac')):
            # Construct Input Path
            input_path = os.path.join(root, file)
            
            # Construct Output Path (mirroring structure)
            # Replace input root with output root
            relative_path = os.path.relpath(input_path, INPUT_DIR)
            output_path = os.path.join(OUTPUT_DIR, relative_path)
            
            # Change extension to .png
            output_path = os.path.splitext(output_path)[0] + '.png'
            
            # Create directory if it doesn't exist
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            
            # Generate
            create_spectrogram(input_path, output_path)

print("Preprocessing Complete. You can now use this output as a dataset.")

KeyboardInterrupt: 